### Preprocessing Yeast Matrix From Microarray Expression Data

#### 0. Misc Explorations

##### GWEIGHTs

In [ ]:
import os
import pandas as pd

# Directory containing .pcl files
pcl_directory = '/home/logs/jtorresb/yeastformer/yeast/yeast_data/all_pcls'

# List to store file names with non-uniform GWEIGHT values
files_with_non_uniform_gweight = []

# Iterate over each .pcl file in the directory
for file_name in os.listdir(pcl_directory):
    if file_name.endswith(".pcl"):
        file_path = os.path.join(pcl_directory, file_name)
        try:
            # Load the .pcl file
            df = pd.read_csv(file_path, sep="\t", index_col=0)

            # Check if the 'GWEIGHT' column exists
            if 'GWEIGHT' in df.columns:
                # Check if all values in 'GWEIGHT' are equal to 1
                if not (df['GWEIGHT'] == 1).all():
                    files_with_non_uniform_gweight.append(file_name)

        except Exception as e:
            print(f"Error processing file '{file_name}': {e}")

# Print the results
if files_with_non_uniform_gweight:
    print("Files with non-uniform GWEIGHT values:")
    for file in files_with_non_uniform_gweight:
        print(file)
else:
    print("All files have GWEIGHT values uniformly equal to 1.")

##### Total Number of Columns

In [ ]:
import os
import pandas as pd

# Directory containing .pcl files
pcl_directory = '/home/logs/jtorresb/Geneformer/yeast/yeast_data/all_pcls'

# Counter for total number of experiment columns
total_experiment_columns = 0

# Iterate over each .pcl file in the directory
for file_name in os.listdir(pcl_directory):
    if file_name.endswith(".pcl"):
        file_path = os.path.join(pcl_directory, file_name)
        try:
            # Load the .pcl file
            df = pd.read_csv(file_path, sep="\t", index_col=0)

            # Count the experiment columns (excluding 'GWEIGHT', 'NAME', 'IDENTIFIER', 'Description', etc.)
            experiment_columns = [col for col in df.columns if col not in ['GWEIGHT', 'NAME', 'IDENTIFIER', 'Description']]
            total_experiment_columns += len(experiment_columns)

        except Exception as e:
            print(f"Error processing file '{file_name}': {e}")

# Print the total number of experiment columns
print(f"Total number of experiment columns across all files: {total_experiment_columns}")

##### Files with More Genes than Genome

In [ ]:
import os
import pandas as pd

# Directory containing .pcl files
pcl_directory = '/home/logs/jtorresb/Geneformer/yeast/yeast_data/all_pcls'

# Threshold for the size of the yeast genome
threshold = 7337

# Counter for files with unique rows exceeding the threshold
count_exceeding_files = 0

# Iterate over each .pcl file in the directory
for file_name in os.listdir(pcl_directory):
    if file_name.endswith(".pcl"):
        file_path = os.path.join(pcl_directory, file_name)
        try:
            # Load the .pcl file
            df = pd.read_csv(file_path, sep="\t", index_col=0)

            # Find the number of unique rows
            unique_count = df.drop_duplicates().shape[0]

            # Check if the unique count exceeds the threshold
            if unique_count > threshold:
                count_exceeding_files += 1

                # Count rows where the YORF (index) starts with 'SGD'
                sgd_count = sum(df.index.astype(str).str.startswith('SGD'))

                print(f"File: {file_name}, Number of unique rows: {unique_count}")

        except Exception as e:
            print(f"Error processing file '{file_name}': {e}")

# Print the total count of files exceeding the threshold
print(f"Number of files with unique rows larger than {threshold}: {count_exceeding_files}")

#### 1. Normalizing Columns in Each .pcl File (Old - Now Normalizing after Merge)

In [ ]:
import os
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Directory containing .pcl files
pcl_directory = '/home/logs/jtorresb/Geneformer/yeast/yeast_data/all_pcls'

files_processed = 0

# Iterate over each .pcl file in the directory
for file_name in os.listdir(pcl_directory):
    if file_name.endswith(".pcl"):
        file_path = os.path.join(pcl_directory, file_name)
        try:
            # Load the .pcl file
            df = pd.read_csv(file_path, sep="\t", index_col=0)

            # Exclude non-experiment columns from normalization
            experiment_columns = [col for col in df.columns if col not in ['GWEIGHT', 'NAME']]

            # Cast the experiment columns to float32
            df[experiment_columns] = df[experiment_columns].astype('float32')

            # Initialize the scaler
            scaler = StandardScaler()

            # Apply z-score normalization for each experiment column
            df[experiment_columns] = scaler.fit_transform(df[experiment_columns])

            # Save the modified DataFrame back to the same file
            df.to_csv(file_path, sep="\t")

            files_processed += 1

        except Exception as e:
            print(f"Error processing file '{file_name}': {e}")

print(f"Processed {files_processed} files.")

#### 2. Ensuring Valid Systematic Names (YORFs) are the Index

##### Dealing with SGD Indexes

In [ ]:
import os
import pandas as pd

# Directory containing .pcl files and the all_yeast_genes.tsv file
pcl_directory = '/home/logs/jtorresb/yeastformer/yeast/yeast_data/dual_channel_pcls_modified'
genes_file = '/home/logs/jtorresb/yeastformer/yeast/yeast_data/genes_info/all_yeast_genes.tsv'

# Load the all_yeast_genes.tsv to create the mapping dictionary
genes_df = pd.read_csv(genes_file, sep='\t')
gene_mapping = dict(zip(genes_df['Gene > Primary DBID'], genes_df['Gene > Systematic Name']))

files_processed = 0

# Iterate over each .pcl file in the directory
for file_name in os.listdir(pcl_directory):
    if file_name.endswith(".pcl"):
        file_path = os.path.join(pcl_directory, file_name)
        try:
            # Load the .pcl file
            df = pd.read_csv(file_path, sep="\t", index_col=0)

            # Replace index values that start with 'SGD' using the mapping dictionary
            df.index = df.index.to_series().apply(lambda x: gene_mapping.get(x, x) if x.startswith('SGD') else x)

            # Save the modified DataFrame back to the same file
            df.to_csv(file_path, sep="\t")

            files_processed += 1

        except Exception as e:
            print(f"Error processing file '{file_name}': {e}")

print(f"Processed {files_processed} files.")

In [ ]:
import os
import pandas as pd

# Directory containing .pcl files
pcl_directory = '/home/logs/jtorresb/yeastformer/yeast/yeast_data/dual_channel_pcls_modified'

files_checked = 0
sgd_found = 0

# Iterate over each .pcl file in the directory
for file_name in os.listdir(pcl_directory):
    if file_name.endswith(".pcl"):
        file_path = os.path.join(pcl_directory, file_name)
        try:
            # Load the .pcl file
            df = pd.read_csv(file_path, sep="\t", index_col=0)

            # Check if any index starts with 'SGD'
            if df.index.str.startswith('SGD').any():
                print(f"Found 'SGD' in indices of file: {file_name}")
                sgd_found += 1

            files_checked += 1

        except Exception as e:
            print(f"Error processing file '{file_name}': {e}")

print(f"Checked {files_checked} files.")
print(f"Found 'SGD' indices in {sgd_found} files.")

##### Dealing with Standard Indexes

In [ ]:
import os
import pandas as pd

# Paths
pcl_directory = '/home/logs/jtorresb/yeastformer/yeast/yeast_data/dual_channel_pcls_modified'
yeast_genes_file = '/home/logs/jtorresb/yeastformer/yeast/yeast_data/genes_info/all_yeast_genes_rest_of_problematic_update.tsv'
# /home/logs/jtorresb/yeastformer/yeast/yeast_data/genes_info/all_yeast_genes_rest_of_problematic_update.tsv --- for some manual fixes

# Load the yeast genes data
yeast_genes_df = pd.read_csv(yeast_genes_file, sep='\t')

# Create a dictionary mapping Gene > Standard Name to Gene > Systematic Name (case-insensitive)
standard_to_systematic = {
    standard.upper(): systematic.upper() 
    for standard, systematic in zip(
        yeast_genes_df['Gene > Standard Name'].dropna(), 
        yeast_genes_df['Gene > Systematic Name'].dropna()
    )
}

# Iterate over each .pcl file in the directory
for file_name in os.listdir(pcl_directory):
    if file_name.endswith(".pcl"):
        pcl_file_path = os.path.join(pcl_directory, file_name)
        try:
            # Load the PCL file
            pcl_df = pd.read_csv(pcl_file_path, sep='\t', index_col=0)

            # Flag to check if the file was modified
            modified = False

            # Create a new index list
            new_index = []
            for index in pcl_df.index:
                # Check if the index is problematic (not in standard_to_systematic dictionary)
                upper_index = index.upper()
                if upper_index not in standard_to_systematic:
                    # Keep the index as is if no mapping exists
                    new_index.append(index)
                else:
                    # Replace the index with the corresponding Gene > Systematic Name
                    new_index.append(standard_to_systematic[upper_index])
                    modified = True

            # Update the index of the DataFrame if modified
            if modified:
                pcl_df.index = new_index

                # Overwrite the original file
                pcl_df.to_csv(pcl_file_path, sep='\t')

                print(f"File '{file_name}' updated successfully.")

        except Exception as e:
            print(f"Error processing file '{file_name}': {e}")

##### Dealing with the Rest

In [ ]:
import os
import pandas as pd
from collections import Counter

# Paths
pcl_directory = '/home/logs/jtorresb/yeastformer/yeast/yeast_data/dual_channel_pcls_modified'
yeast_genes_file = '/home/logs/jtorresb/yeastformer/yeast/yeast_data/genes_info/all_yeast_genes.tsv'

# Load the yeast genes data
yeast_genes_df = pd.read_csv(yeast_genes_file, sep='\t')

# Create sets for faster lookups
valid_systematic_names = set(yeast_genes_df['Gene > Systematic Name'])

problematic_gene_counts = Counter()

# Iterate over each .pcl file in the directory
for file_name in os.listdir(pcl_directory):
    if file_name.endswith(".pcl"):
        pcl_file_path = os.path.join(pcl_directory, file_name)
        try:
            # Load the PCL file
            pcl_df = pd.read_csv(pcl_file_path, sep='\t', index_col=0)

            # Find problematic indexes
            problematic_indexes = [
                index for index in pcl_df.index
                if index not in valid_systematic_names
            ]

            # Print problematic file and indexes
            # if problematic_indexes:
                # print(f"File: {file_name}")
                # print(f"Problematic indexes: {problematic_indexes}")
                # print('-----------------------------------')

            # Update the counter with the problematic indexes
            problematic_gene_counts.update(problematic_indexes)

        except Exception as e:
            print(f"Error processing file '{file_name}': {e}")

# Iterate over all genes, not just the top 100, and filter those with count < 10
for gene, count in problematic_gene_counts.items():
    if count < 10 and count >=5 :
        print(f"{gene}: {count} occurrences")


###### For now, let's remove the problematic index if YORF available and same values

In [ ]:
import os
import pandas as pd

# Paths to directories and files
pcl_directory = '/home/logs/jtorresb/yeastformer/yeast/yeast_data/dual_channel_pcls_modified'
gene_mapping_path = '/home/logs/jtorresb/yeastformer/yeast/yeast_data/genes_info/all_yeast_genes.tsv'

# Load the valid systematic names from the mapping file
gene_mapping_df = pd.read_csv(gene_mapping_path, sep='\t')
valid_systematic_names = set(gene_mapping_df['Gene > Systematic Name'].dropna())

# Process each .pcl file
for file_count, file_name in enumerate(os.listdir(pcl_directory), start=1):
    if file_name.endswith(".pcl"):
        file_path = os.path.join(pcl_directory, file_name)
        try:
            # Load the PCL file
            pcl_df = pd.read_csv(file_path, sep='\t', index_col=0)

            # Identify problematic indexes (not in valid systematic names)
            problematic_indexes = [
                index for index in pcl_df.index if index not in valid_systematic_names
            ]

            if problematic_indexes:
                print(f"Processing file ({file_count}/{len(os.listdir(pcl_directory))}): {file_name}")
                print(f"Problematic indexes: {problematic_indexes}")

                # Check if problematic indexes have matching valid systematic name rows
                rows_to_drop = []
                for problem_index in problematic_indexes:
                    # Check if there is a matching row with valid systematic name
                    matching_rows = pcl_df.loc[
                        pcl_df.index.isin(valid_systematic_names) & 
                        (pcl_df.loc[problem_index].drop(['NAME', 'GWEIGHT'], errors='ignore') == pcl_df.drop(['NAME', 'GWEIGHT'], axis=1, errors='ignore')).all(axis=1)
                    ]

                    # If a matching row exists, mark the problematic index for removal
                    if not matching_rows.empty:
                        rows_to_drop.append(problem_index)

                # Remove the problematic rows and overwrite the file
                if rows_to_drop:
                    pcl_df = pcl_df.drop(index=rows_to_drop)
                    pcl_df.to_csv(file_path, sep='\t')
                    print(f"Fixed and overwritten file ({file_count}/{len(os.listdir(pcl_directory))}): {file_name}")
                else:
                    print(f"No matching rows found for problematic indexes in file ({file_count}/{len(os.listdir(pcl_directory))}): {file_name}")

        except Exception as e:
            print(f"Error processing file ({file_count}/{len(os.listdir(pcl_directory))}): {file_name}: {e}")

###### Apparently, most of the remaining problematic genes are "LTRs". Remove them

In [ ]:
import os
import pandas as pd

# Paths to directories and files
pcl_directory = '/home/logs/jtorresb/yeastformer/yeast/yeast_data/dual_channel_pcls_modified'

# Process each .pcl file
for file_count, file_name in enumerate(os.listdir(pcl_directory), start=1):
    if file_name.endswith(".pcl"):
        file_path = os.path.join(pcl_directory, file_name)
        try:
            # Load the PCL file
            pcl_df = pd.read_csv(file_path, sep='\t', index_col=0)

            # Identify rows to remove by checking if index contains 'delta', 'sigma', 'tau' or 'omega'
            rows_to_remove = [
                index for index in pcl_df.index if any(x in index.lower() for x in ['delta', 'sigma', 'tau', 'omega'])
            ]

            if rows_to_remove:
                print(f"Processing file ({file_count}/{len(os.listdir(pcl_directory))}): {file_name}")
                print(f"Rows to remove: {rows_to_remove}")

                # Remove the rows and overwrite the file
                pcl_df = pcl_df.drop(index=rows_to_remove)
                pcl_df.to_csv(file_path, sep='\t')
                print(f"Fixed and overwritten file ({file_count}/{len(os.listdir(pcl_directory))}): {file_name}")

        except Exception as e:
            print(f"Error processing file ({file_count}/{len(os.listdir(pcl_directory))}): {file_name}: {e}")

###### Removing more Retrotransposons

In [ ]:
import os
import pandas as pd

# Paths to directories and files
pcl_directory = '/home/logs/jtorresb/yeastformer/yeast/yeast_data/dual_channel_pcls_modified'

# List of gene names to remove
genes_to_remove = [
    "YGRWTy2-2", "YLRCTy2-2", "YLRCTy1-1", "YMRCTy1-4", "YORCTy2-1", "YPLCTy4-1", "YPLWTy1-1",
    "YILWTy3-1", "YGRCTy1-3", "YBRWTy1-2", "YDRWTy1-5", "YFLWTy2-1", "YLRWTy1-3", "YOLWTy1-1",
    "YLRWTy1-2", "YGRWTy1-1", "YBLWTy1-1", "YPRWTy1-3", "YDRCTy1-3", "YJLWTy4-1", "YNLCTy2-1",
    "YJRWTy1-2", "YDRCTy1-1", "YPRCTy1-4", "YPRCTy1-2", "YMLWTy1-2", "YDRCTy2-1", "YARCTy1-1",
    "YDRWTy2-2", "YNLWTy1-2", "YNLCTy1-1", "YDRWTy1-4", "YGRCTy1-2", "YBLWTy2-1", "YORWTy2-2",
    "YCLWTy2-1", "YORWTy1-2", "YGRWTy3-1", "YDRWTy2-3", "YDRCTy1-2", "YJRWTy1-1", "YHRCTy1-1",
    "YMLWTy1-1", "YERCTy1-1", "YMRCTy1-3", "YLRWTy2-1", "YGRCTy2-1", "YCLWTy5-1'"
]

# Process each .pcl file
for file_count, file_name in enumerate(os.listdir(pcl_directory), start=1):
    if file_name.endswith(".pcl"):
        file_path = os.path.join(pcl_directory, file_name)
        try:
            # Load the PCL file
            pcl_df = pd.read_csv(file_path, sep='\t', index_col=0)

            # Identify rows to remove by checking if index is in the list of genes to remove
            rows_to_remove = [
                index for index in pcl_df.index if index in genes_to_remove
            ]

            if rows_to_remove:
                print(f"Processing file ({file_count}/{len(os.listdir(pcl_directory))}): {file_name}")
                print(f"Rows to remove: {rows_to_remove}")

                # Remove the rows and overwrite the file
                pcl_df = pcl_df.drop(index=rows_to_remove)
                pcl_df.to_csv(file_path, sep='\t')
                print(f"Fixed and overwritten file ({file_count}/{len(os.listdir(pcl_directory))}): {file_name}")
            else:
                print(f"No rows to remove in file ({file_count}/{len(os.listdir(pcl_directory))}): {file_name}")

        except Exception as e:
            print(f"Error processing file ({file_count}/{len(os.listdir(pcl_directory))}): {file_name}: {e}")


###### Remove the remaining ones for simplicity, which do not seem relevant (eg MIS11 different organism)

In [ ]:
import os
import pandas as pd

# Paths
pcl_directory = '/home/logs/jtorresb/yeastformer/yeast/yeast_data/dual_channel_pcls_modified'
yeast_genes_file = '/home/logs/jtorresb/yeastformer/yeast/yeast_data/genes_info/all_yeast_genes.tsv'

# Load the yeast genes data
yeast_genes_df = pd.read_csv(yeast_genes_file, sep='\t')

# Get the set of valid systematic names, ensuring case insensitivity
valid_systematic_names = set(yeast_genes_df['Gene > Systematic Name'].str.upper())

# Iterate over each .pcl file in the directory
for file_name in os.listdir(pcl_directory):
    if file_name.endswith(".pcl"):
        pcl_file_path = os.path.join(pcl_directory, file_name)
        try:
            # Load the PCL file
            pcl_df = pd.read_csv(pcl_file_path, sep='\t', index_col=0)

            # Identify rows with indexes not in the valid systematic names
            invalid_indexes = [index for index in pcl_df.index if index.upper() not in valid_systematic_names]

            # If invalid rows are found, drop them
            if invalid_indexes:
                pcl_df.drop(index=invalid_indexes, inplace=True)

                # Overwrite the original file
                pcl_df.to_csv(pcl_file_path, sep='\t')
                print(f"File '{file_name}' updated: Removed {len(invalid_indexes)} invalid rows.")

        except Exception as e:
            print(f"Error processing file '{file_name}': {e}")

##### Final Check

In [ ]:
import os
import pandas as pd

# Paths
pcl_directory = '/home/logs/jtorresb/yeastformer/yeast/yeast_data/dual_channel_pcls_modified'
yeast_genes_file = '/home/logs/jtorresb/yeastformer/yeast/yeast_data/genes_info/all_yeast_genes.tsv'

# Load the yeast genes data
yeast_genes_df = pd.read_csv(yeast_genes_file, sep='\t')

# Get the set of valid systematic names, ensuring case insensitivity
valid_systematic_names = set(yeast_genes_df['Gene > Systematic Name'].str.upper())

# Iterate over each .pcl file in the directory
for file_name in os.listdir(pcl_directory):
    if file_name.endswith(".pcl"):
        pcl_file_path = os.path.join(pcl_directory, file_name)
        try:
            # Load the PCL file
            pcl_df = pd.read_csv(pcl_file_path, sep='\t', index_col=0)

            # Identify rows with indexes not in the valid systematic names
            invalid_indexes = [index for index in pcl_df.index if index.upper() not in valid_systematic_names]

            # If invalid rows are found, report them
            if invalid_indexes:
                print(f"File '{file_name}' has {len(invalid_indexes)} invalid rows: {', '.join(invalid_indexes)}.")
            # else:
            #     print(f"File '{file_name}' has all valid rows.")

        except Exception as e:
            print(f"Error processing file '{file_name}': {e}")

#### 3. Dealing with Duplicates

In [ ]:
import os
import pandas as pd

# Paths
pcl_directory = '/home/logs/jtorresb/yeastformer/yeast/yeast_data/dual_channel_pcls_modified'

# Iterate over each .pcl file in the directory
for file_name in os.listdir(pcl_directory):
    if file_name.endswith(".pcl"):
        pcl_file_path = os.path.join(pcl_directory, file_name)
        try:
            # Load the PCL file
            pcl_df = pd.read_csv(pcl_file_path, sep='\t', index_col=0)
            
            # Identify duplicate indexes
            duplicate_indexes = pcl_df.index[pcl_df.index.duplicated()].unique()
            
            if len(duplicate_indexes) > 1000:
                print(f"File: {file_name}")
                print(f"Number of duplicate indexes: {len(duplicate_indexes)}")
        
        except Exception as e:
            print(f"Error processing file '{file_name}': {e}")


##### Direct duplicates

In [ ]:
import os
import pandas as pd

# Directory containing PCL files
pcl_directory = '/home/logs/jtorresb/yeastformer/yeast/yeast_data/dual_channel_pcls_modified'

# Iterate over each .pcl file in the directory
for file_name in os.listdir(pcl_directory):
    if file_name.endswith(".pcl"):
        pcl_file_path = os.path.join(pcl_directory, file_name)

        try:
            # Load the PCL file
            pcl_df = pd.read_csv(pcl_file_path, sep='\t', index_col=0)

            # Rename index to "YORF"
            pcl_df.index.name = "YORF"

            # Identify experimental columns (excluding NAME and GWEIGHT)
            experimental_cols = [col for col in pcl_df.columns if col not in ["NAME", "GWEIGHT"]]

            # Reset index to include the YORF in duplicate detection
            pcl_df_reset = pcl_df.reset_index()

            # Detect exact duplicates (same YORF and experimental values)
            duplicate_mask = pcl_df_reset.duplicated(subset=["YORF"] + experimental_cols, keep=False)

            # Filter duplicated rows
            duplicate_df = pcl_df_reset[duplicate_mask]

            if not duplicate_df.empty:
                print(f"\nDuplicates found in {file_name}:")
                duplicate_genes = duplicate_df["YORF"].unique()
                print(f"Duplicate genes: {', '.join(duplicate_genes)}")
                #print(duplicate_df)

        except Exception as e:
            print(f"❌ Error processing file '{file_name}': {e}")

In [ ]:
import os
import pandas as pd

# Directory containing PCL files
pcl_directory = '/home/logs/jtorresb/yeastformer/yeast/yeast_data/dual_channel_pcls_modified'

# Iterate over each .pcl file in the directory
for file_name in os.listdir(pcl_directory):
    if file_name.endswith(".pcl"):
        pcl_file_path = os.path.join(pcl_directory, file_name)

        try:
            # Load the PCL file
            pcl_df = pd.read_csv(pcl_file_path, sep='\t', index_col=0)

            # Rename index to "YORF"
            pcl_df.index.name = "YORF"

            # Identify experimental columns (excluding NAME and GWEIGHT)
            experimental_cols = [col for col in pcl_df.columns if col not in ["NAME", "GWEIGHT"]]

            # Reset index for duplicate detection
            pcl_df_reset = pcl_df.reset_index()

            # Drop exact duplicates (same index and same experimental values)
            pcl_df_cleaned = pcl_df_reset.drop_duplicates(subset=["YORF"] + experimental_cols, keep="first")

            # Save back to the original file
            pcl_df_cleaned.to_csv(pcl_file_path, sep='\t', index=False)

            print(f"✅ Cleaned duplicates from {file_name}")

        except Exception as e:
            print(f"❌ Error processing file '{file_name}': {e}")

##### Pseudoduplicates

###### Aux

In [ ]:
import os
import pandas as pd

# Paths
pcl_directory = '/home/logs/jtorresb/yeastformer/yeast/yeast_data/dual_channel_pcls_modified'

# Track first pseudoduplicate group
first_pseudoduplicate_printed = False

# Iterate over all .pcl files
for file_name in os.listdir(pcl_directory):
    if file_name.endswith(".pcl"):
        pcl_file_path = os.path.join(pcl_directory, file_name)
        
        try:
            # Load the PCL file
            pcl_df = pd.read_csv(pcl_file_path, sep='\t', index_col=0)
            
            # Identify experimental columns (excluding NAME and GWEIGHT)
            experimental_cols = [col for col in pcl_df.columns if col not in ["NAME", "GWEIGHT"]]
            
            # Count pseudoduplicates
            duplicate_groups = pcl_df.groupby(pcl_df.index)

            pseudoduplicate_count = 0
            first_pseudoduplicate_family = None

            for yorf, group in duplicate_groups:
                if len(group) > 1:
                    unique_experiment_values = group[experimental_cols].drop_duplicates()
                    
                    if len(unique_experiment_values) > 1:
                        pseudoduplicate_count += 1
                        
                        # Store the first pseudoduplicate group if not printed yet
                        if not first_pseudoduplicate_printed:
                            first_pseudoduplicate_family = group
                            first_pseudoduplicate_printed = None
            
            # Print count for the file if there are pseudoduplicates
            if pseudoduplicate_count > 0:
                print(f"File: {file_name} - Pseudoduplicate YORFs: {pseudoduplicate_count}")
                
                # Print the first found pseudoduplicate family
                if first_pseudoduplicate_family is not None:
                    print("\nFirst Pseudoduplicate Family:")
                    print(first_pseudoduplicate_family.index)
                    print("-----------------------------------")

        except Exception as e:
            print(f"Error processing file '{file_name}': {e}")

In [ ]:
import os
import pandas as pd

# Paths
pcl_directory = '/home/logs/jtorresb/yeastformer/yeast/yeast_data/all_pcls_modified'

done = 0

# Iterate over all .pcl files
for file_name in os.listdir(pcl_directory):
    if done:
        break
    if file_name.endswith(".pcl"):
        pcl_file_path = os.path.join(pcl_directory, file_name)
        
        try:
            # Load the PCL file
            pcl_df = pd.read_csv(pcl_file_path, sep='\t', index_col=0)
            
            # Identify experimental columns (excluding NAME and GWEIGHT)
            experimental_cols = [col for col in pcl_df.columns if col not in ["NAME", "GWEIGHT"]]
            
            # Count pseudoduplicates
            duplicate_groups = pcl_df.groupby(pcl_df.index)
            
            for gene, group in duplicate_groups:
                if len(group) >= 2: 
                    print(f"File: {file_name} - Gene: {gene}")
                    done = True
        
        except Exception as e:
            print(f"Error processing file '{file_name}': {e}")

###### Fixing the Issue

In [20]:
import os
import pandas as pd
import numpy as np
from scipy.stats import pearsonr

def load_pcl(file_path):
    """Loads a .pcl file into a Pandas DataFrame, using the first column as the index."""
    try:
        df = pd.read_csv(file_path, sep='\t', index_col=0)
        df.drop(columns=['NAME', 'GWEIGHT'], errors='ignore', inplace=True)  # Ignore if not present
        return df
    except Exception as e:
        print(f"Error loading file '{file_path}': {e}")
        return None

def compute_correlation_matrix(group):
    """Computes the correlation matrix for a group of pseudoduplicates."""
    return group.T.corr()

def merge_pseudoduplicates(group, threshold=0.8):
    """Merges correlated pseudoduplicates and selects one if uncorrelated remain."""
    if len(group) == 1:
        return group  # Only one row, no duplicates
    
    corr_matrix = compute_correlation_matrix(group)
    merged_rows = []
    used = set()
    
    for i in range(len(group)):
        if i in used:
            continue
        correlated = [i]
        
        for j in range(i + 1, len(group)):
            if j not in used and corr_matrix.iloc[i, j] >= threshold:
                correlated.append(j)
                used.add(j)
        
        merged_rows.append(group.iloc[correlated].mean())
        used.add(i)
    
    if len(merged_rows) > 1:
        return pd.DataFrame([merged_rows[np.random.choice(len(merged_rows))]])  # Pick one randomly
    
    return pd.DataFrame(merged_rows)  # Return merged row

def process_pcl_files(pcl_directory, output_directory):
    """Processes all .pcl files in the directory, cleans pseudoduplicates, and saves results."""
    os.makedirs(output_directory, exist_ok=True)
    
    for filename in os.listdir(pcl_directory):
        if filename.endswith(".pcl"):
            print(f"Processing: {filename}")
            file_path = os.path.join(pcl_directory, filename)
            df = load_pcl(file_path)
            
            if df is None:
                continue
            
            # Identify experimental columns
            experimental_cols = [col for col in df.columns if col not in ["NAME", "GWEIGHT"]]
            df = df[experimental_cols]  # Keep only experimental columns
            
            cleaned_data = []
            duplicate_groups = df.groupby(df.index)
            
            for gene, group in duplicate_groups:
                cleaned_group = merge_pseudoduplicates(group)
                cleaned_group.index = [gene] * len(cleaned_group)  # Keep original index
                cleaned_data.append(cleaned_group)
            
            cleaned_df = pd.concat(cleaned_data)
            output_path = os.path.join(output_directory, filename)
            cleaned_df.to_csv(output_path, sep='\t')
            print(f"Processed: {filename} -> {output_path}")

# Set directories
pcl_directory = '/home/logs/jtorresb/yeastformer/yeast/yeast_data/dual_channel_pcls_modified'
output_directory = pcl_directory

# Run processing
process_pcl_files(pcl_directory, output_directory)

Processing: Chechik_2008_PMID_18953355_GSE13219_set10_family.pcl
Processed: Chechik_2008_PMID_18953355_GSE13219_set10_family.pcl -> /home/logs/jtorresb/yeastformer/yeast/yeast_data/dual_channel_pcls_modified/Chechik_2008_PMID_18953355_GSE13219_set10_family.pcl
Processing: Gasch_2000_PMID_11102521_2010.Gasch00_HSto37.flt.knn.avg.pcl
Processed: Gasch_2000_PMID_11102521_2010.Gasch00_HSto37.flt.knn.avg.pcl -> /home/logs/jtorresb/yeastformer/yeast/yeast_data/dual_channel_pcls_modified/Gasch_2000_PMID_11102521_2010.Gasch00_HSto37.flt.knn.avg.pcl
Processing: Zhou_2011_PMID_21700227_GSE23580_final.pcl
Processed: Zhou_2011_PMID_21700227_GSE23580_final.pcl -> /home/logs/jtorresb/yeastformer/yeast/yeast_data/dual_channel_pcls_modified/Zhou_2011_PMID_21700227_GSE23580_final.pcl
Processing: Kaplan_2008_PMID_19023413_GSE12822_setA_family.pcl
Processed: Kaplan_2008_PMID_19023413_GSE12822_setA_family.pcl -> /home/logs/jtorresb/yeastformer/yeast/yeast_data/dual_channel_pcls_modified/Kaplan_2008_PMID_19

####  Aux: Removing unnecesary metadata

In [19]:
import os
import pandas as pd

# Directory containing the modified .pcl files
pcl_dir = "/home/logs/jtorresb/yeastformer/yeast/yeast_data/dual_channel_pcls_modified"

# Columns to remove (case-insensitive for 'description')
columns_to_remove = ['GWEIGHT', 'NAME', 'IDENTIFIER', 'GENE', 'Name']

for file_name in os.listdir(pcl_dir):
    if file_name.endswith(".pcl"):
        file_path = os.path.join(pcl_dir, file_name)
        try:
            # Read the file
            df = pd.read_csv(file_path, sep="\t", header=0, index_col=0)

            # Normalize column names for case-insensitive 'Description' detection
            columns_lower = [col.lower() for col in df.columns]
            description_cols = [col for col, low in zip(df.columns, columns_lower) if low == "description"]

            # Full list of columns to remove (including case variants of 'Description')
            full_remove_list = columns_to_remove + description_cols

            # Drop the columns if they exist
            df = df.drop(columns=[col for col in full_remove_list if col in df.columns])

            # Overwrite the file
            df.to_csv(file_path, sep="\t")
            print(f"Removed specified columns from: {file_name}")

        except Exception as e:
            print(f"Error processing {file_name}: {e}")

print("Finished removing specified columns.")


Removed specified columns from: Chechik_2008_PMID_18953355_GSE13219_set10_family.pcl
Removed specified columns from: Gasch_2000_PMID_11102521_2010.Gasch00_HSto37.flt.knn.avg.pcl
Removed specified columns from: Zhou_2011_PMID_21700227_GSE23580_final.pcl
Removed specified columns from: Kaplan_2008_PMID_19023413_GSE12822_setA_family.pcl
Removed specified columns from: Lu_2022_PMID_36066422_GSE137261.final.pcl
Removed specified columns from: Venkatasubrahmanyam_2007_PMID_17925448_GSE4826_setA_family.pcl
Removed specified columns from: Cullen_2004_PMID_15256499_2010.glycosylation.pcl
Removed specified columns from: Chechik_2008_PMID_18953355_GSE13219_set11_family.pcl
Removed specified columns from: Burrill_2011_PMID_21363961_GSE36210.final.pcl
Removed specified columns from: Joseph-Strauss_2007_PMID_17999778_GSE7393_set1_family.pcl
Removed specified columns from: Gasch_2000_PMID_11102521_2010.Gasch00_steadyState(y13).flt.knn.avg.pcl
Removed specified columns from: Friedlander_2006_PMID_1654

#### Aux: Averaging out replicates

In [ ]:
import os
import re
import pandas as pd
from tqdm import tqdm 

def extract_base(col_name):
    """
    Extracts the base condition name from a column name by removing a replicate marker.
    Recognizes patterns like:
      - rep1, rep 1, rep_1, replicate 1, replicate_1, Replicate, etc.
    even when followed by extra text.
    """
    pattern = re.compile(r"^(.*?)\s*(rep(?:licate)?\s*[_-]?\s*\d+)(.*)$", re.IGNORECASE)
    m = pattern.match(col_name)
    if m:
        # Concatenate the part before the replicate marker with the part after it.
        base = (m.group(1) + m.group(3)).strip()
        if base:
            return base
    return col_name

def format_float(x):
    """Format floats with 6 significant digits; otherwise return the value unchanged."""
    if isinstance(x, float):
        return format(x, ".6g")
    return x

In [ ]:
# List of annotation row identifiers to exclude from averaging.
annotation_keys = ['EWEIGHT', 'NAME', 'GWEIGHT']

# Path to a single .pcl file for testing.
file_path = "/home/logs/jtorresb/yeastformer/yeast/yeast_data/data_inspection/dual_channel_extracted/Renaud-Young_2015_PMID_25701288/GSE66176.final.pcl"

try:
    # Read the file into a DataFrame (assuming tab-delimited and the first column is the index)
    df = pd.read_csv(file_path, sep="\t", header=0, index_col=0)

    # Separate annotation rows from the data rows.
    annotations = df.loc[df.index.isin(annotation_keys)]
    data = df.loc[~df.index.isin(annotation_keys)]

    # Build a dictionary grouping columns by their "base" name.
    groups = {}
    for col in data.columns:
        base = extract_base(col)
        groups.setdefault(base, []).append(col)

    # Identify if any averaging is needed.
    needs_averaging = any(len(cols) > 1 for cols in groups.values())

    if not needs_averaging:
        print("No replicates present. No averaging was performed.")
    else:
        # Create a new DataFrame for the averaged expression data.
        new_data = pd.DataFrame(index=data.index)
        for base, cols in groups.items():
            if len(cols) > 1:
                new_data[base] = data[cols].mean(axis=1)
            else:
                new_data[base] = data[cols[0]]
        
        # **Reindex annotations so that they use the new averaged column names:**
        annotations = annotations.reindex(columns=new_data.columns)

        # Combine the annotations with the averaged data.
        new_df = pd.concat([annotations, new_data])
    
        # Format all float values to 6 significant digits using Series.map.
        for col in new_df.columns:
            new_df[col] = new_df[col].map(format_float)
    
        # Overwrite the original file or write to a new file for testing.
        new_df.to_csv(file_path, sep="\t")
        print("File processed and written successfully.")
    
    # # For debugging/testing: Print the first few rows of the new dataframe.
    # print(new_df.head())

except Exception as e:
    print(f"Error processing {file_path}: {e}")

File processed and written successfully.


In [ ]:
import os
import re
import pandas as pd
from tqdm import tqdm

# Annotation row identifiers to retain and handle specially.
annotation_keys = ['EWEIGHT', 'NAME', 'GWEIGHT']

def extract_base(col_name):
    pattern = re.compile(r"^(.*?)\s*(?:rep(?:licate)?|repeat)\s*[_-]?\s*\d+(.*)$", re.IGNORECASE)
    m = pattern.match(col_name)
    if m:
        base = (m.group(1) + m.group(2)).strip()
        return base if base else col_name
    return col_name

# Format function for floats.
def format_float(x):
    return format(x, ".6g") if isinstance(x, float) else x

# Root directory with subfolders containing .pcl files.
root_dir = "/home/logs/jtorresb/yeastformer/yeast/yeast_data/data_inspection/dual_channel_extracted"

# List all subfolders.
subfolders = [sub for sub in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, sub))]

# Process each .pcl file in all subfolders.
for subfolder in tqdm(subfolders, desc="Processing subfolders"):
    subfolder_path = os.path.join(root_dir, subfolder)
    pcl_files = [f for f in os.listdir(subfolder_path) if f.endswith(".pcl")]

    for file_name in pcl_files:
        file_path = os.path.join(subfolder_path, file_name)
        try:
            df = pd.read_csv(file_path, sep="\t", header=0, index_col=0)

            annotations = df.loc[df.index.isin(annotation_keys)]
            data = df.loc[~df.index.isin(annotation_keys)]

            groups = {}
            for col in data.columns:
                base = extract_base(col)
                groups.setdefault(base, []).append(col)

            if not any(len(cols) > 1 for cols in groups.values()):
                continue  # No averaging needed

            # Create averaged data.
            new_data = pd.DataFrame(index=data.index)
            for base, cols in groups.items():
                if len(cols) > 1:
                    new_data[base] = data[cols].mean(axis=1)
                else:
                    new_data[base] = data[cols[0]]

            # Create new annotation DataFrame with values from the first column of each group.
            new_annotations = pd.DataFrame(index=annotations.index, columns=new_data.columns)
            for base, cols in groups.items():
                first_col = cols[0]
                for key in annotation_keys:
                    if key in annotations.index:
                        new_annotations.at[key, base] = annotations.at[key, first_col]

            # Concatenate annotations + averaged expression data
            new_df = pd.concat([new_annotations, new_data])

            # Format floats
            new_df = new_df.applymap(format_float)

            # Save result
            new_df.to_csv(file_path, sep="\t")
        
        except Exception as e:
            print(f"Error processing {file_path}: {e}")

print("✅ Processing complete.")

Processing subfolders:   0%|          | 0/309 [00:00<?, ?it/s]/tmp/ipykernel_2375784/1260748054.py:65: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  new_df = pd.concat([new_annotations, new_data])
/tmp/ipykernel_2375784/1260748054.py:68: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  new_df = new_df.applymap(format_float)
Processing subfolders:  16%|█▌        | 49/309 [00:01<00:07, 32.84it/s]/tmp/ipykernel_2375784/1260748054.py:65: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the c

✅ Processing complete.
